# Social Media Activity & Donation Prediction
Predicting how social media activity drives donation volume and value.

## 1. Problem Framing
Social media campaigns (posts, stories, fundraising links) generate donor traffic and one-time gifts. This model quantifies the relationship between social media engagement metrics and downstream donation outcomes, and predicts the expected donation amount attributable to a given post.

**Target variable:** `donations_amount` — continuous (total USD donated within 7 days of a social media post).

**Secondary classification target:** `post_converted` — binary (1 = post generated at least one donation within 7 days).

**Success metric (regression):** R² ≥ 0.60, RMSE as low as feasible.

**Success metric (classification):** ROC-AUC ≥ 0.75.

**Stakeholders:** Communications / marketing team, development staff.

## 2. Data Acquisition & Preparation

In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

np.random.seed(42)
n = 1000

platforms = ['Instagram', 'Facebook', 'Twitter', 'LinkedIn', 'TikTok']
content_types = ['Story', 'Photo', 'Video', 'Link', 'Reel']

df = pd.DataFrame({
    'post_id': range(1, n + 1),
    'platform': np.random.choice(platforms, size=n, p=[0.30, 0.30, 0.15, 0.15, 0.10]),
    'content_type': np.random.choice(content_types, size=n),
    'reach': np.random.randint(100, 50000, size=n),
    'impressions': np.random.randint(100, 80000, size=n),
    'likes': np.random.randint(0, 2000, size=n),
    'comments': np.random.randint(0, 300, size=n),
    'shares': np.random.randint(0, 500, size=n),
    'link_clicks': np.random.randint(0, 1000, size=n),
    'hour_of_day': np.random.randint(0, 24, size=n),
    'day_of_week': np.random.randint(0, 7, size=n),  # 0=Mon
    'follower_count': np.random.randint(500, 50000, size=n),
    'has_donate_button': np.random.randint(0, 2, size=n),
    'campaign_active': np.random.randint(0, 2, size=n),
})

df = pd.get_dummies(df, columns=['platform', 'content_type'], drop_first=True)

# Simulate donation amount
base = (
    20
    + 0.002 * df['link_clicks']
    + 0.001 * df['shares'] * 5
    + 50 * df['has_donate_button']
    + 30 * df['campaign_active']
    + np.random.exponential(20, size=n)
).clip(0)
df['donations_amount'] = base * np.random.lognormal(0, 0.5, size=n)
df['post_converted'] = (df['donations_amount'] > 30).astype(int)

print(df.shape)
print(f"Conversion rate: {df['post_converted'].mean():.2%}")
df.head()

In [ ]:
feature_cols = [
    'reach', 'impressions', 'likes', 'comments', 'shares', 'link_clicks',
    'hour_of_day', 'day_of_week', 'follower_count',
    'has_donate_button', 'campaign_active'
] + [c for c in df.columns if c.startswith('platform_') or c.startswith('content_type_')]

X = df[feature_cols]
y_reg = df['donations_amount']
y_cls = df['post_converted']

X_train, X_test, y_reg_train, y_reg_test, y_cls_train, y_cls_test = train_test_split(
    X, y_reg, y_cls, test_size=0.2, random_state=42
)

scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_test_sc = scaler.transform(X_test)

print(f"Train: {X_train.shape}, Test: {X_test.shape}")

## 3. Exploration

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Donation amount distribution (log scale)
axes[0].hist(np.log1p(df['donations_amount']), bins=30, color='steelblue', edgecolor='white')
axes[0].set_title('log(1 + Donation Amount)')
axes[0].set_xlabel('log(1 + USD)')

# Link clicks vs donation amount
axes[1].scatter(df['link_clicks'], df['donations_amount'], alpha=0.3, s=8, color='purple')
axes[1].set_xlabel('Link Clicks')
axes[1].set_ylabel('Donation Amount (USD)')
axes[1].set_title('Link Clicks vs Donation Amount')

# Average donation by day of week
day_names = ['Mon', 'Tue', 'Wed', 'Thu', 'Fri', 'Sat', 'Sun']
avg_by_day = df.groupby('day_of_week')['donations_amount'].mean()
axes[2].bar(day_names, avg_by_day.values, color='teal')
axes[2].set_title('Avg Donation by Day of Week')
axes[2].set_ylabel('Avg USD')

plt.tight_layout()
plt.show()

## 4. Modeling
### 4a. Predictive Model — Gradient Boosting Regressor (donation amount) + Classifier (conversion)
### 4b. Causal / Explanatory Model — SHAP (TreeExplainer on regressor)

In [ ]:
from sklearn.ensemble import GradientBoostingRegressor, GradientBoostingClassifier
import numpy as np

# --- 4a: Predictive regression model (log-transform target for stability) ---
y_reg_train_log = np.log1p(y_reg_train)
y_reg_test_log = np.log1p(y_reg_test)

gbr = GradientBoostingRegressor(n_estimators=200, max_depth=4, learning_rate=0.05, random_state=42)
gbr.fit(X_train, y_reg_train_log)

y_pred_log = gbr.predict(X_test)
y_pred_amount = np.expm1(y_pred_log)

# Predictive classifier model
gbc = GradientBoostingClassifier(n_estimators=150, max_depth=3, learning_rate=0.05, random_state=42)
gbc.fit(X_train, y_cls_train)
y_cls_proba = gbc.predict_proba(X_test)[:, 1]
y_cls_pred = gbc.predict(X_test)

print("Regressor and classifier training complete.")

In [ ]:
# --- 4b: Causal / Explanatory model using SHAP on the regressor ---
try:
    import shap
    explainer = shap.TreeExplainer(gbr)
    shap_values = explainer.shap_values(X_test)

    shap.summary_plot(shap_values, X_test, feature_names=feature_cols, show=False)
    plt.title('SHAP Summary — Social Media Donation Amount (log scale)')
    plt.tight_layout()
    plt.show()

    # Dependence plot: link_clicks effect
    if 'link_clicks' in feature_cols:
        idx = feature_cols.index('link_clicks')
        shap.dependence_plot(
            idx, shap_values, X_test,
            feature_names=feature_cols, show=False
        )
        plt.title('SHAP Dependence — link_clicks')
        plt.tight_layout()
        plt.show()
except ImportError:
    print("shap not installed — skipping. Install with: pip install shap")

## 5. Evaluation

In [ ]:
from sklearn.metrics import (
    r2_score, mean_squared_error,
    roc_auc_score, classification_report,
    RocCurveDisplay
)

# Regression metrics
r2 = r2_score(y_reg_test_log, y_pred_log)
rmse_log = mean_squared_error(y_reg_test_log, y_pred_log, squared=False)
rmse_usd = mean_squared_error(y_reg_test, y_pred_amount, squared=False)
print(f"Regression — R² (log): {r2:.4f}, RMSE (log): {rmse_log:.4f}, RMSE (USD): ${rmse_usd:.2f}")
print()

# Classification metrics
print(f"Classification ROC-AUC: {roc_auc_score(y_cls_test, y_cls_proba):.4f}")
print(classification_report(y_cls_test, y_cls_pred, target_names=['No Conversion', 'Conversion']))

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].scatter(y_reg_test_log, y_pred_log, alpha=0.3, s=8, color='steelblue')
mn, mx = y_reg_test_log.min(), y_reg_test_log.max()
axes[0].plot([mn, mx], [mn, mx], 'r--')
axes[0].set_xlabel('Actual log(1 + Amount)')
axes[0].set_ylabel('Predicted log(1 + Amount)')
axes[0].set_title(f'Predicted vs Actual (R²={r2:.3f})')

RocCurveDisplay.from_predictions(y_cls_test, y_cls_proba, ax=axes[1], name='GBC')
axes[1].set_title('ROC Curve — Post Conversion')

plt.tight_layout()
plt.show()

## 6. Feature Selection

In [ ]:
importances = pd.Series(gbr.feature_importances_, index=feature_cols).sort_values(ascending=False)

plt.figure(figsize=(10, 4))
importances.head(12).plot(kind='bar', color='steelblue')
plt.title('Feature Importances — Social Media Donation Regressor')
plt.ylabel('Importance')
plt.tight_layout()
plt.show()

selected_features = importances[importances >= importances.mean()].index.tolist()
print("Selected features:", selected_features)

## 7. Deployment

In [ ]:
import pickle, os

os.makedirs('models', exist_ok=True)

X_train_sel = X_train[selected_features]
X_test_sel = X_test[selected_features]

gbr_final = GradientBoostingRegressor(
    n_estimators=200, max_depth=4, learning_rate=0.05, random_state=42
)
gbr_final.fit(X_train_sel, y_reg_train_log)

with open('models/social_media_donations_model.pkl', 'wb') as f:
    pickle.dump({'model': gbr_final, 'features': selected_features, 'scaler': scaler}, f)

print("Model saved to models/social_media_donations_model.pkl")

# Forecast expected donations for upcoming posts
upcoming_posts = X_test_sel.iloc[:5].copy()
upcoming_posts['predicted_donation_usd'] = np.expm1(gbr_final.predict(upcoming_posts))
print("\nExpected donations for upcoming posts:")
print(upcoming_posts[['predicted_donation_usd']].round(2))